직렬화 가능 여부 확인하기

In [ ]:
# [목적] OpenAI 모델과 프롬프트를 준비해, 이후에 저장할 수 있는 체인을 만드는 예제입니다.
# API 키를 불러온 뒤 과일 이름을 질문에 넣는 프롬프트를 만들며, 다음 셀에서 모델과 연결합니다.

import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
# LangSmith에 체인 실행 과정과 결과를 기록하는 도구입니다.
from langchain_teddynote import logging

# .env 파일이 있으면 API 키를 현재 실행 환경에 불러옵니다.
load_dotenv()
logging.langsmith("Chapter7-Modesl")

prompt = PromptTemplate.from_template("{fruit}의 색상이 무엇입니까?")

In [ ]:
# [목적] ChatOpenAI 클래스 자체가 저장 가능한 형식으로 바뀔 수 있는지 확인하는 예제입니다.
# True이면 설정 정보를 직렬화해 파일이나 데이터 형태로 다룰 수 있는 기반이 됩니다.

print(f"ChatOpenAI: {ChatOpenAI.is_lc_serializable()}")

In [ ]:
# [목적] 사용할 OpenAI 모델 객체를 만들고, 이 객체도 저장 가능한지 확인하는 예제입니다.
# 모델 이름과 temperature 설정은 체인에 포함되어 직렬화할 때 함께 저장됩니다.

# temperature=0은 답변을 더 일관되게 생성하도록 설정합니다.
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

print(f"ChatOpenAI: {llm.is_lc_serializable()}")

In [ ]:
# [목적] 프롬프트와 모델을 하나의 체인으로 연결해, 저장하고 다시 불러올 대상을 만드는 예제입니다.
# | 연산자는 프롬프트 결과를 모델 입력으로 전달하며, 만들어진 체인은 다음 셀에서 직렬화합니다.

# 프롬프트와 모델을 연결해 입력부터 답변 생성까지 처리하는 체인을 만듭니다.
chain = prompt | llm

chain.is_lc_serializable()

체인 직렬화 하기

In [ ]:
# [목적] 체인의 설정을 파이썬 딕셔너리와 JSON 문자열 형태로 바꾸는 예제입니다.
# 이 과정이 직렬화이며, 변환 결과는 파일에 저장하거나 다른 환경으로 전달할 때 사용합니다.

# LangChain 객체를 딕셔너리나 JSON 문자열로 바꾸는 직렬화 함수입니다.
from langchain_core.load import dumpd, dumps

dumpd_chain = dumpd(chain)
dumpd_chain

In [ ]:
# [목적] dumpd()로 변환한 결과의 자료형을 확인하는 예제입니다.
# 딕셔너리인지 확인하면 이후 pickle이나 JSON 파일로 저장할 수 있음을 알 수 있습니다.

type(dumpd_chain)

In [ ]:
# [목적] 같은 체인을 JSON 형식의 문자열로 직렬화하는 예제입니다.
# 문자열 형태는 텍스트로 전달하거나 JSON 형식이 필요한 곳에 저장할 때 편리합니다.

dumps_chain = dumps(chain)
dumps_chain

In [ ]:
# [목적] dumps()로 변환한 결과가 문자열인지 확인하는 예제입니다.
# dumpd()의 딕셔너리 결과와 비교해 두 직렬화 방식의 차이를 확인합니다.

type(dumps_chain)

Pickle 파일로 직렬화하고 로드하기

In [ ]:
# [목적] 딕셔너리로 직렬화한 체인을 Pickle 파일로 저장하는 예제입니다.
# 파일을 만들면 다음 실행이나 다른 노트북에서도 체인 설정을 다시 불러올 수 있습니다.

import pickle

# fruit_chain.pkl 파일로 직렬화된 체인을 저장
with open("fruit_chain.pkl", "wb") as f:
    pickle.dump(dumpd_chain, f)

In [ ]:
# [목적] 같은 체인 설정을 사람이 읽기 쉬운 JSON 파일로 저장하는 예제입니다.
# JSON 파일은 구조를 확인하기 쉽고, 마지막 셀에서 다시 읽어 체인으로 복원할 수 있습니다.

import json

with open("fruit_chain.json", "w") as fp:
    json.dump(dumpd_chain, fp)

In [ ]:
# [목적] Pickle 파일에 저장한 체인 설정을 메모리로 다시 읽어 오는 예제입니다.
# 이 단계에서는 파일 데이터를 읽기만 하며, 다음 셀에서 실제 LangChain 체인 객체로 복원합니다.

import pickle

with open("fruit_chain.pkl", "rb") as f:  # Pickle 파일을 로드
    loaded_chain = pickle.load(f)

In [ ]:
# [목적] Pickle에서 읽은 설정 데이터를 다시 실행 가능한 LangChain 체인으로 복원하는 예제입니다.
# 복원한 체인에 API 키를 연결하고 사과를 입력해, 저장 전과 동일하게 작동하는지 확인합니다.

# 직렬화된 설정을 LangChain 객체로 되돌리는 역직렬화 함수입니다.
from langchain_core.load import load

# 본인이 저장한 OpenAI 체인을 허용하고, 실행에 필요한 API 키를 다시 연결합니다.
chain_from_file = load(loaded_chain, secrets_map={"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]}, allowed_objects="all")  # 저장한 체인을 로드

print(chain_from_file.invoke({"fruit": "사과"}))  # 체인을 실행

In [ ]:
# [목적] API 키를 지정하면서 저장된 딕셔너리 데이터를 체인으로 복원하는 예제입니다.
# API 키처럼 저장하지 않는 비밀 정보는 secrets_map으로 연결한 뒤, 복원된 체인을 실행합니다.

# load는 딕셔너리, loads는 JSON 문자열을 역직렬화할 때 사용하는 함수입니다.
from langchain_core.load import load, loads

load_chain = load(
    loaded_chain, secrets_map={"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]}, allowed_objects="all"
)

load_chain.invoke({"fruit": "사과"})  # 불러온 체인이 정상 동작하는지 확인

In [ ]:
# [목적] JSON 파일에 저장한 체인 설정을 읽어 다시 실행 가능한 체인으로 복원하는 예제입니다.
# 파일을 딕셔너리로 읽고 API 키를 연결한 뒤, 저장 전과 같은 질문을 실행해 결과를 확인합니다.

with open("fruit_chain.json", "r") as fp:
    loaded_from_json_chain = json.load(fp)
    loads_chain = load(loaded_from_json_chain, secrets_map={"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]}, allowed_objects="all")

loads_chain.invoke({"fruit": "사과"})

### 직렬화와 역직렬화

- **직렬화**: 체인처럼 프로그램 안에 있는 객체를 딕셔너리·JSON·파일처럼 저장하거나 전달할 수 있는 형태로 바꾸는 일입니다.
- **역직렬화**: 저장한 데이터를 다시 읽어, 실행 가능한 체인 객체로 되돌리는 일입니다.

이 노트북에서는 체인을 파일로 직렬화한 뒤, API 키를 연결해 다시 역직렬화하고 실행합니다.